In [1]:
!unzip data_set.zip

Archive:  data_set.zip
   creating: data_set/
   creating: data_set/Non_Poisnous_Edible/
   creating: data_set/Non_Poisnous_Edible/almond_mushroom/
  inflating: data_set/Non_Poisnous_Edible/almond_mushroom/almond_mushroom_001.png  
  inflating: data_set/Non_Poisnous_Edible/almond_mushroom/almond_mushroom_002.png  
  inflating: data_set/Non_Poisnous_Edible/almond_mushroom/almond_mushroom_003.png  
  inflating: data_set/Non_Poisnous_Edible/almond_mushroom/almond_mushroom_004.png  
  inflating: data_set/Non_Poisnous_Edible/almond_mushroom/almond_mushroom_005.png  
  inflating: data_set/Non_Poisnous_Edible/almond_mushroom/almond_mushroom_006.png  
  inflating: data_set/Non_Poisnous_Edible/almond_mushroom/almond_mushroom_007.png  
  inflating: data_set/Non_Poisnous_Edible/almond_mushroom/almond_mushroom_008.png  
  inflating: data_set/Non_Poisnous_Edible/almond_mushroom/almond_mushroom_009.png  
  inflating: data_set/Non_Poisnous_Edible/almond_mushroom/almond_mushroom_010.png  
  inflating:

In [2]:
!pip install efficientnet_pytorch


  Preparing metadata (setup.py) ... done
  Created wheel for efficientnet_pytorch: filename=efficientnet_pytorch-0.7.1-py3-none-any.whl size=16426 sha256=4df7e0ade4e9992bbc5a04bfaa1960030902b343becf9f4b354ccec6f95f7260
  Stored in directory: /root/.cache/pip/wheels/9c/3f/43/e6271c7026fe08c185da2be23c98c8e87477d3db63f41f32ad
Successfully built efficientnet_pytorch


In [4]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from efficientnet_pytorch import EfficientNet
from PIL import Image
from sklearn.model_selection import train_test_split
import torch.nn.functional as F

DATA_DIR = "data_set"


In [5]:
class MushroomDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = root
        self.transform = transform

        self.main_classes = sorted(os.listdir(root))
        self.main_to_idx = {m:i for i,m in enumerate(self.main_classes)}

        self.samples = []
        self.sub_classes = []
        self.sub_to_idx = {}
        sub_index = 0

        for main in self.main_classes:
            main_path = os.path.join(root, main)
            subclasses = sorted(os.listdir(main_path))

            for sub in subclasses:
                sub_path = os.path.join(main_path, sub)

                if sub not in self.sub_to_idx:
                    self.sub_to_idx[sub] = sub_index
                    self.sub_classes.append(sub)
                    sub_index += 1

                for img_name in os.listdir(sub_path):
                    img_path = os.path.join(sub_path, img_name)
                    self.samples.append((img_path,
                                         self.main_to_idx[main],
                                         self.sub_to_idx[sub]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, main_label, sub_label = self.samples[idx]
        img = Image.open(path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, main_label, sub_label


In [6]:
IMG_SIZE = 300

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(25),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

val_tfms = transforms.Compose([
    transforms.Resize(IMG_SIZE + 20),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])


In [7]:
full_dataset = MushroomDataset(DATA_DIR, transform=train_tfms)

indices = list(range(len(full_dataset)))
train_idx, val_idx = train_test_split(indices, test_size=0.15, random_state=42)

train_subset = torch.utils.data.Subset(full_dataset, train_idx)
val_subset = torch.utils.data.Subset(full_dataset, val_idx)
val_subset.dataset.transform = val_tfms

BATCH_SIZE = 16

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

NUM_MAIN = len(full_dataset.main_classes)
NUM_SUB = len(full_dataset.sub_classes)

print("Main classes:", NUM_MAIN)
print("Sub classes:", NUM_SUB)


Main classes: 4
Sub classes: 215


In [8]:
class HierarchicalEffNetB3(nn.Module):
    def __init__(self, num_main, num_sub):
        super().__init__()
        # Using EfficientNet-B3 backbone
        self.backbone = EfficientNet.from_pretrained('efficientnet-b3')

        in_features = self.backbone._fc.in_features
        self.backbone._fc = nn.Identity()  # Remove default classifier

        self.main_head = nn.Linear(in_features, num_main)
        self.sub_head  = nn.Linear(in_features, num_sub)

    def forward(self, x):
        feat = self.backbone(x)
        return self.main_head(feat), self.sub_head(feat)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = HierarchicalEffNetB3(NUM_MAIN, NUM_SUB).to(device)

criterion_main = nn.CrossEntropyLoss()
criterion_sub  = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)


Downloading: "https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b3-5fb5a3c3.pth" to /root/.cache/torch/hub/checkpoints/efficientnet-b3-5fb5a3c3.pth


100%|██████████| 47.1M/47.1M [00:00<00:00, 175MB/s]


Loaded pretrained weights for efficientnet-b3


In [9]:
def evaluate():
    model.eval()
    correct_main, correct_sub, total = 0, 0, 0

    with torch.no_grad():
        for imgs, m_lbl, s_lbl in val_loader:
            imgs, m_lbl, s_lbl = imgs.to(device), m_lbl.to(device), s_lbl.to(device)
            m_out, s_out = model(imgs)

            correct_main += (m_out.argmax(1) == m_lbl).sum().item()
            correct_sub  += (s_out.argmax(1) == s_lbl).sum().item()
            total += m_lbl.size(0)

    return correct_main / total, correct_sub / total


In [10]:
EPOCHS = 80      # Recommended for EfficientNet-B3
PATIENCE = 10    # Early stopping patience

best_sub_acc = 0
patience_counter = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for imgs, m_lbl, s_lbl in train_loader:
        imgs, m_lbl, s_lbl = imgs.to(device), m_lbl.to(device), s_lbl.to(device)

        m_out, s_out = model(imgs)
        loss = criterion_main(m_out, m_lbl) + criterion_sub(s_out, s_lbl)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    val_main_acc, val_sub_acc = evaluate()

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Loss: {total_loss/len(train_loader):.4f} | "
          f"Main Acc: {val_main_acc:.4f} | Sub Acc: {val_sub_acc:.4f}")

    if val_sub_acc > best_sub_acc:
        best_sub_acc = val_sub_acc
        patience_counter = 0
        torch.save(model.state_dict(), "best_hierarchical_effnet_b3.pth")
        print("✔ Model improved — saved!")
    else:
        patience_counter += 1
        print(f"⚠ No improvement. Patience: {patience_counter}/{PATIENCE}")

    if patience_counter >= PATIENCE:
        print("Early stopping triggered!")
        break


Epoch 1/80 | Loss: 6.3396 | Main Acc: 0.6205 | Sub Acc: 0.1215
✔ Model improved — saved!
Epoch 2/80 | Loss: 5.3070 | Main Acc: 0.6716 | Sub Acc: 0.2921
✔ Model improved — saved!
Epoch 3/80 | Loss: 4.1271 | Main Acc: 0.7036 | Sub Acc: 0.3710
✔ Model improved — saved!
Epoch 4/80 | Loss: 3.1115 | Main Acc: 0.7335 | Sub Acc: 0.4712
✔ Model improved — saved!
Epoch 5/80 | Loss: 2.3099 | Main Acc: 0.7292 | Sub Acc: 0.5352
✔ Model improved — saved!
Epoch 6/80 | Loss: 1.7049 | Main Acc: 0.7783 | Sub Acc: 0.5864
✔ Model improved — saved!
Epoch 7/80 | Loss: 1.2517 | Main Acc: 0.7655 | Sub Acc: 0.6034
✔ Model improved — saved!
Epoch 8/80 | Loss: 0.9034 | Main Acc: 0.7420 | Sub Acc: 0.6333
✔ Model improved — saved!
Epoch 9/80 | Loss: 0.6612 | Main Acc: 0.7612 | Sub Acc: 0.6525
✔ Model improved — saved!
Epoch 10/80 | Loss: 0.4978 | Main Acc: 0.7569 | Sub Acc: 0.6631
✔ Model improved — saved!
Epoch 11/80 | Loss: 0.3921 | Main Acc: 0.7676 | Sub Acc: 0.6652
✔ Model improved — saved!
Epoch 12/80 | Loss:

In [16]:
def predict(model, img_path, dataset, device="cpu"):
    model.eval()

    tfm = transforms.Compose([
        transforms.Resize(320),
        transforms.CenterCrop(300),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])

    # Load the image
    img = Image.open(img_path).convert("RGB")
    x = tfm(img).unsqueeze(0).to(device)

    with torch.no_grad():
        m_out, s_out = model(x)

        # Convert to probabilities (0–1)
        m_probs = F.softmax(m_out, dim=1)
        s_probs = F.softmax(s_out, dim=1)

        # Get highest probability class
        m_prob, m_idx = torch.max(m_probs, 1)
        s_prob, s_idx = torch.max(s_probs, 1)

        # Convert probabilities to percent (0–100)
        m_prob = float(m_prob.item() * 100)
        s_prob = float(s_prob.item() * 100)

        return {
            "main_class": dataset.main_classes[m_idx.item()],
            "main_probability_percent": m_prob,
            "species": dataset.sub_classes[s_idx.item()],
            "species_probability_percent": s_prob
        }


In [20]:
result = predict(model, "2.jpg", full_dataset, device)
print(result)


{'main_class': 'Non_Poisnous_Edible', 'main_probability_percent': 99.8324453830719, 'species': 'chicken_of_the_woods', 'species_probability_percent': 39.73707854747772}
